# F1 Trivia RAG — experimental run

End-to-end smoke test of the chatbot: ingest one real season of Ergast race results,
build the Gemini-embedded Chroma index, then ask the citation-aware query engine a
few questions and inspect the sources behind each answer.

Scope is kept to a single season (2023) to keep the ingestion + embedding calls fast.

> Outputs are cleared. The stored ones were produced before season-aware retrieval and
> before the migration off `google.generativeai`, so they showed the undercount this
> project exists to fix ("Red Bull won five races in 2023" — the old `top_k` of 5).
> Re-run the notebook to regenerate them.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)  # so f1_trivia_rag.config picks up ./.env
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"Project root: {PROJECT_ROOT}")

## 1. Ingest

Pull every 2023 race result from the Ergast (jolpica) API and normalize into `RawDocument`s.

In [ ]:
from f1_trivia_rag.ingestion.ergast import fetch_season_results

documents = fetch_season_results(2023)
print(f"Fetched {len(documents)} race-result documents")
print(documents[0].source_id)
print(documents[0].text)

## 2. Build the index

Embeds every document with Gemini (`models/gemini-embedding-001`) and persists to the
project's Chroma store at `storage/chroma` (same location `scripts/ingest.py` uses).

In [ ]:
from f1_trivia_rag.rag.build_index import build_index

index = build_index(documents)
print("Index built and persisted to storage/chroma")

## 3. Load the query engine and chat

Same `load_query_engine()` the FastAPI `/chat` endpoint uses — citation-aware, and
season-aware: a question naming a season is retrieved with a `season` filter and a
`top_k` sized to that season's stored nodes, so aggregate questions see every race
rather than the five nearest chunks.

In [ ]:
from f1_trivia_rag.rag.query_engine import load_query_engine

engine = load_query_engine()

In [ ]:
def ask(question: str) -> None:
    response = engine.query(question)
    print(f"Q: {question}")
    print(f"A: {response}\n")
    print("Citations:")
    for node in response.source_nodes:
        meta = node.metadata
        print(f"  - {meta.get('source')}:{meta.get('source_id')} ({meta.get('race_name')})")
    print("-" * 60)

In [ ]:
ask("Who won the 2023 Monaco Grand Prix?")

In [ ]:
ask("Which constructor won the most races in the 2023 season?")

In [ ]:
ask("Did any driver fail to finish the 2023 Australian Grand Prix, and why?")